# Kaggle survey v3 — E4 + E5 — ổn định chọn gen (TCGA, GSE37745)

Notebook sinh bởi `scripts/make_kaggle_notebooks.py` — **không sửa tay**; chỉ sửa ô cấu hình trước mỗi phiên.
Notebook chỉ chọn việc và gọi `scripts/kaggle_jobs.py` (M18). Hướng dẫn chung: `docs/survey_v2/KAGGLE_RUN_PLAN.md`.

## Thí nghiệm

E4 (M = 30 tập con) trên TCGA và GSE37745; E5a dư thừa, E5b nhạy seed (rf_importance, stability_selection mọi k; GA, BPSO k = 10), E5c trùng lặp ở k = 10 — chỉ GSE37745 (E5c có cả TCGA). Đầu ra `E4_E5/<bộ>/`.

## Các việc (mỗi phiên Kaggle = một JOB)

| JOB | Đợt · hạn | Khôi phục (Input) | Giờ | Nội dung |
|---|---|---|---|---|
| `e4e5_gse37745` | 2 · 06/11 | `e1_phaseA_geo` | 4–8 | E4 (M = 30) + E5a + E5b + E5c trên GSE37745 (+GA, BPSO ở k = 10); chạy lại sau wrap_gse37745 để E5c có GA/BPSO (phần đã có được bỏ qua) |
| `e4_tcga` | 2 · 06/11 | `e1_phaseB_tcga` | 6–11 | E4 (M = 30) + E5c trên TCGA (+GA, BPSO ở k = 10); quá 12 giờ thì chạy lại, tự tiếp tục |

## Thứ tự phiên

1. **Đợt 2 (06/11):** `e4e5_gse37745` (khôi phục output `e1_phaseA_geo` của notebook E1) và `e4_tcga` (khôi phục output `e1_phaseB_tcga`); hai phiên chạy song song được.
2. `e4_tcga` có thể quá 12 giờ → phiên dừng mã 124, chạy lại cùng JOB với output vừa lưu.
3. **Đợt 3 (08/11):** chạy lại `e4e5_gse37745`, khôi phục output lần 1 + `wrap_gse37745` (notebook E1) — chỉ bổ sung E5c có GA/BPSO (vài phút).

## Mỗi phiên

1. Add Input: **experiment-data-v3** (bản 29/10 sau QC) + output các việc ở cột *Khôi phục*.
2. Ô cấu hình: `JOB`, `RESTORE_INPUTS`, `REPO_COMMIT` (giữ một commit cho mọi phiên của một đợt).
3. Save Version → Save & Run All. Output: `/kaggle/working/<JOB>_results.zip` → lưu thành Dataset `kltn-<job>`.
4. Mã 124 (quá `MAX_HOURS`): phiên mới cùng JOB, khôi phục zip vừa lưu — script tự tiếp tục từ checkpoint.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time, zipfile

JOBS_IN_NOTEBOOK = ['e4e5_gse37745', 'e4_tcga']
JOB = 'e4e5_gse37745'  # chọn một trong JOBS_IN_NOTEBOOK (bảng ở đầu notebook)
INPUT_V3_DATASET = Path('/kaggle/input/experiment-data-v3')   # sửa theo slug Kaggle Input của bạn
RESTORE_INPUTS = []            # thư mục Kaggle Input chứa <job>_results.zip của các việc trước, vd. [Path('/kaggle/input/e1-phasea-geo')]
REPO_URL = 'https://github.com/datachain-uit/RNA-Seq-FS-Survey.git'
REPO_BRANCH = 'survey-v2-code'
REPO_COMMIT = None             # vd. 'bc9171c' — ghim một commit (đã push) cho mọi phiên của một đợt; None = đầu nhánh
MAX_HOURS = 11.0               # dừng có kiểm soát trước giới hạn 12 giờ của Kaggle
PARALLEL = None                # None = theo việc (thường 4 = số nhân CPU của Kaggle)
RUN_TESTS = True               # pytest nhanh (~1 phút) trước khi chạy: môi trường Kaggle có chạy được code không

WORK_ROOT = Path('/kaggle/working/kltn_data')
REPO_DIR = Path('/kaggle/working/RNA-Seq-FS-Survey')
LOG_DIR = Path('/kaggle/working/logs')
T0 = time.time()
assert JOB in JOBS_IN_NOTEBOOK, f'{JOB} không thuộc notebook này — dùng notebook của thí nghiệm tương ứng'

In [ ]:
# Gate 1 — Kaggle Input đúng layout v3 và là bản sau QC (có excluded_samples, bản chia few-shot có cột subset).
DATASETS = {'tcga_luad_lusc_primary_tumor': 'experiment-data-perm', 'gse10245': 'experiment-data-geo-target',
            'gse30219': 'experiment-data-geo-target', 'gse37745': 'experiment-data-geo-target', 'gse42127': 'experiment-data-geo-target'}
assert INPUT_V3_DATASET.is_dir(), f'Không thấy Kaggle Input: {INPUT_V3_DATASET}'
for folder in ('experiment-data-perm', 'experiment-data-geo-target', 'split_manifests'):
    assert (INPUT_V3_DATASET / folder).is_dir(), f'Thiếu {folder}'
for name, branch in DATASETS.items():
    d = INPUT_V3_DATASET / branch / name
    for f in ('expression.h5', 'metadata.csv', 'repeated_splits.csv', f'excluded_samples_{name}.csv'):
        assert (d / f).is_file(), f'Thiếu {name}/{f} — Kaggle Input chưa phải bản 29/10 sau QC'
    if branch == 'experiment-data-geo-target':
        header = (d / 'fewshot_splits.csv').open(encoding='utf-8-sig').readline()
        assert 'subset' in header, f'{name}/fewshot_splits.csv là bản cũ (thiếu cột subset)'
assert (INPUT_V3_DATASET / 'experiment-data-perm' / 'tcga_normal_vs_tumor_sanity' / 'expression.h5').is_file(), 'Thiếu bộ sanity'
print('INPUT_V3_OK')

In [ ]:
# Gate 2 — clone đúng commit đã push; cài phần thiếu trên ảnh Kaggle (không cài requirements.txt để không làm hỏng numpy/scipy của Kaggle).
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
if REPO_COMMIT:
    subprocess.run(['git', 'checkout', '--quiet', REPO_COMMIT], cwd=REPO_DIR, check=True)
commit_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
pushed = subprocess.run(['git', 'branch', '-r', '--contains', 'HEAD'], cwd=REPO_DIR, capture_output=True, text=True).stdout.strip()
assert pushed, f'{commit_hash} không có trên remote — kết quả sẽ không truy vết được'
for pkg in (['skrebate==0.8.4'], ['abess']):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', *pkg], check=False)
for mod in ('numpy', 'pandas', 'scipy', 'sklearn', 'tables', 'joblib', 'threadpoolctl', 'psutil', 'skrebate', 'abess', 'matplotlib'):
    try:
        print(f'{mod:14s}', getattr(__import__(mod), '__version__', 'ok'))
    except Exception as exc:     # abess thiếu -> phương pháp abess tự bị bỏ qua và ghi vào config (ghi HANDOVER)
        print(f'{mod:14s} KHÔNG IMPORT ĐƯỢC: {exc}')
print('COMMIT', commit_hash)

In [ ]:
# Gate 3 — dữ liệu read-only liên kết vào work root; khôi phục output của các việc trước; biến môi trường.
WORK_ROOT.mkdir(parents=True, exist_ok=True)
for folder in ('experiment-data-perm', 'experiment-data-geo-target', 'split_manifests'):
    link = WORK_ROOT / folder
    if not link.exists(): link.symlink_to(INPUT_V3_DATASET / folder, target_is_directory=True)
restored = []
for src in map(Path, RESTORE_INPUTS):
    assert src.exists(), f'Không thấy Input khôi phục: {src}'
    zips = [src] if src.suffix == '.zip' else sorted(src.rglob('*_results.zip'))
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall(WORK_ROOT, members=[m for m in zf.namelist() if m.startswith('results/')])
        restored.append(str(z))
    if not zips and (src / 'results').is_dir():       # output đã được Kaggle giải nén
        shutil.copytree(src / 'results', WORK_ROOT / 'results', dirs_exist_ok=True)
        restored.append(str(src / 'results'))
print('RESTORED', restored or '—')
os.environ.update({'KLTN_DATA_ROOT': str(WORK_ROOT), 'KLTN_STRICT_N': '1', 'KLTN_LOG_DIR': str(LOG_DIR), 'PYTHONUTF8': '1'})
os.environ.pop('KLTN_ALLOW_UNPUSHED', None)
subprocess.run([sys.executable, 'scripts/kaggle_jobs.py', 'show', JOB], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, 'scripts/kaggle_jobs.py', 'status'], cwd=REPO_DIR, check=True)

In [ ]:
# Kiểm tra nhanh môi trường Kaggle (test đơn vị, không cần dữ liệu thật).
if RUN_TESTS:
    r = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q', '-m', 'not slow', '-p', 'no:cacheprovider', '-x'], cwd=REPO_DIR)
    assert r.returncode == 0, 'pytest không đạt trên Kaggle — không chạy thực nghiệm'

In [ ]:
# Chạy việc. Không raise trước khi lưu output: phần đã chạy (checkpoint) phải được giữ để tiếp tục.
budget = max(0.5, MAX_HOURS - (time.time() - T0) / 3600)
cmd = [sys.executable, 'scripts/kaggle_jobs.py', 'run', JOB, '--max-hours', f'{budget:.2f}', '--log-dir', str(LOG_DIR)]
if PARALLEL: cmd += ['--parallel', str(PARALLEL)]
job_code = subprocess.run(cmd, cwd=REPO_DIR).returncode
print('JOB EXIT', job_code, {0: 'OK', 124: 'TIMEOUT -> chạy lại cùng JOB với output này làm Input', 2: 'thiếu đầu vào',
                             3: 'verify không đạt'}.get(job_code, 'lỗi -> xem logs/'))

In [ ]:
# Lưu output: results/ (gồm phần khôi phục + phần mới) + logs + run_info.json -> /kaggle/working/<JOB>_results.zip
info = {'job': JOB, 'commit_hash': commit_hash, 'exit_code': job_code, 'restored': restored,
        'hours': round((time.time() - T0) / 3600, 2), 'finished_at': time.strftime('%Y-%m-%d %H:%M:%S %Z')}
(WORK_ROOT / 'results').mkdir(exist_ok=True)
(WORK_ROOT / 'results' / f'run_info_{JOB}.json').write_text(json.dumps(info, indent=2), encoding='utf-8')
if LOG_DIR.exists():
    shutil.copytree(LOG_DIR, WORK_ROOT / 'results' / 'logs' / JOB, dirs_exist_ok=True)
archive = Path('/kaggle/working') / f'{JOB}_results.zip'
if archive.exists(): archive.unlink()
shutil.make_archive(str(archive.with_suffix('')), 'zip', root_dir=WORK_ROOT, base_dir='results')
print('ARCHIVE', archive, f'{archive.stat().st_size / 1e6:.1f} MB', info)
assert job_code in (0, 124), f'JOB {JOB} lỗi (mã {job_code}) — xem results/logs/{JOB}/'
if job_code == 124:
    print('CHƯA XONG: tạo phiên mới cùng JOB, thêm output của phiên này vào RESTORE_INPUTS.')